# Week 1 — End-to-end toy dataset test
Verifies: GeoNet FDSN access, ObsPy download, SeisBench + EQTransformer picking.
Toy day: 2019-09-04, the day of the M5.2 Lake Taupō mainshock — chosen deliberately
since it sits inside our revised 1 Aug-30 Sep 2019 analysis window, so this test
doubles as a first look at real data from the actual study period.

Run this in VS Code connected to the HPC via Remote-SSH, with the `tvz-seismic`
conda env selected as the Jupyter kernel (VS Code renders '# %%' blocks as cells).

In [ ]:
import obspy
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import os

OUTPUT_DIR = os.path.expanduser("~/tvz_project/data/toy_test")
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = Client("GEONET")

t0 = UTCDateTime("2019-09-04T00:00:00")
t1 = UTCDateTime("2019-09-05T00:00:00")

In [ ]:
# Pull station inventory for a small handful of stations near Lake Taupō first,
# to confirm which were live on this date before bulk-downloading waveforms.
inventory = client.get_stations(
    network="NZ",
    station="*",
    latitude=-38.8, longitude=176.0, maxradius=1.0,   # ~1 deg radius around Lake Taupo
    starttime=t0, endtime=t1,
    level="channel",
)
print(inventory)

In [ ]:
# Build the channel selection from the actual inventory response.
# Filter to stations that actually have channel data for this date FIRST,
# then take the first 3 — many stations in this radius (older strong-motion
# sites) return an empty channel list for a given date if they weren't
# operating then, and slicing before filtering was picking those by accident.
candidates = []
for net in inventory:
    for sta in net:
        channel_bands = sorted({cha.code[:2] for cha in sta.channels})
        if channel_bands:
            candidates.append((net.code, sta.code, channel_bands))

print(f"{len(candidates)} stations have active channel data for 2019-09-04")

# Prefer broadband (HH) stations where available, for the best pick quality
candidates.sort(key=lambda x: 0 if "HH" in x[2] else 1)
test_stations = candidates[:3]
for net_code, sta_code, channel_bands in test_stations:
    print(f"Selected: {net_code}.{sta_code} — bands {channel_bands}")

st = obspy.Stream()
for net_code, sta_code, channel_bands in test_stations:
    band = "HH" if "HH" in channel_bands else channel_bands[0]
    try:
        st += client.get_waveforms(
            network=net_code, station=sta_code, location="*", channel=f"{band}?",
            starttime=t0, endtime=t1,
        )
    except Exception as e:
        print(f"  {net_code}.{sta_code}: download failed — {e}")

print(st)

if len(st) == 0:
    print("WARNING: no waveform data downloaded — check the channel bands printed above,")
    print("and whether these stations were actually live on 2019-09-04.")
else:
    st.write(os.path.join(OUTPUT_DIR, "toy_2019-09-04.mseed"), format="MSEED")


In [ ]:
# Full-stack test: run SeisBench's EQTransformer over the toy day.
import seisbench
import seisbench.models as sbm

# Known SeisBench issue: intermittent access to the primary model repository
# (https://github.com/seisbench/seisbench#known-issues). Switch to the backup
# mirror before downloading pretrained weights — slower, but more reliable
# from an HPC node. Only affects this download; not a config we need to keep
# permanently once the weights are cached locally.
seisbench.use_backup_repository()

model = sbm.EQTransformer.from_pretrained("original")
model.eval()

# Matching the proposal's stated parameters exactly:
#   - event (detection) probability threshold >= 0.2  -> detection_threshold
#   - phase pick probability threshold        >= 0.01 -> P_threshold / S_threshold
# (Earlier version of this cell incorrectly put 0.2 on P/S instead of detection —
# that mismatch is almost certainly why the previous run returned 0 picks.)
classify_output = model.classify(
    st,
    batch_size=8,
    detection_threshold=0.2,
    P_threshold=0.01,
    S_threshold=0.01,
)

picks = classify_output.picks
detections = classify_output.detections

print(f"Detected {len(detections)} event detections and {len(picks)} phase picks on toy day")
for p in picks[:10]:
    print(p)


In [ ]:
print("=== Week 1 toy test complete ===")
print("If picks were produced above, ObsPy -> SeisBench -> EQTransformer is verified end-to-end.")
print("Next: NonLinLoc + GrowClust3D.jl link-up (Week 1 remaining task) and Week 2's full")
print("station inventory download for the Aug-Sep 2019 window.")